In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 加载数据
df = pd.read_csv('/kaggle/input/online-news-popularity/OnlineNewsPopularity.csv')

# 查看数据基本信息
print(f"数据集形状: {df.shape}")
print("\n前5行数据:")
print(df.head())
print("\n数据描述:")
print(df.describe())
print("\n缺失值检查:")
print(df.isnull().sum())

数据集形状: (39644, 61)

前5行数据:
                                                 url   timedelta  \
0  http://mashable.com/2013/01/07/amazon-instant-...       731.0   
1  http://mashable.com/2013/01/07/ap-samsung-spon...       731.0   
2  http://mashable.com/2013/01/07/apple-40-billio...       731.0   
3  http://mashable.com/2013/01/07/astronaut-notre...       731.0   
4   http://mashable.com/2013/01/07/att-u-verse-apps/       731.0   

    n_tokens_title   n_tokens_content   n_unique_tokens   n_non_stop_words  \
0             12.0              219.0          0.663594                1.0   
1              9.0              255.0          0.604743                1.0   
2              9.0              211.0          0.575130                1.0   
3              9.0              531.0          0.503788                1.0   
4             13.0             1072.0          0.415646                1.0   

    n_non_stop_unique_tokens   num_hrefs   num_self_hrefs   num_imgs  ...  \
0                 

In [2]:
print(df.columns)

Index(['url', ' timedelta', ' n_tokens_title', ' n_tokens_content',
       ' n_unique_tokens', ' n_non_stop_words', ' n_non_stop_unique_tokens',
       ' num_hrefs', ' num_self_hrefs', ' num_imgs', ' num_videos',
       ' average_token_length', ' num_keywords', ' data_channel_is_lifestyle',
       ' data_channel_is_entertainment', ' data_channel_is_bus',
       ' data_channel_is_socmed', ' data_channel_is_tech',
       ' data_channel_is_world', ' kw_min_min', ' kw_max_min', ' kw_avg_min',
       ' kw_min_max', ' kw_max_max', ' kw_avg_max', ' kw_min_avg',
       ' kw_max_avg', ' kw_avg_avg', ' self_reference_min_shares',
       ' self_reference_max_shares', ' self_reference_avg_sharess',
       ' weekday_is_monday', ' weekday_is_tuesday', ' weekday_is_wednesday',
       ' weekday_is_thursday', ' weekday_is_friday', ' weekday_is_saturday',
       ' weekday_is_sunday', ' is_weekend', ' LDA_00', ' LDA_01', ' LDA_02',
       ' LDA_03', ' LDA_04', ' global_subjectivity',
       ' global_sent

In [3]:
# 删除无用和共线性特征
cols_to_drop = ['url', ' timedelta', 
                ' abs_title_sentiment_polarity', ' LDA_04', ' is_weekend',
                ' rate_positive_words', ' abs_title_subjectivity', 
                ' rate_negative_words']

df = df.drop(cols_to_drop, axis=1)

# 回归变量处理 - 已经以one-hot形式存在(data_channel_*, weekday_*)
# 检查回归变量
categorical_cols = [col for col in df.columns if 'data_channel_' in col or 'weekday_' in col]
print(f"回归变量: {categorical_cols}")

回归变量: [' data_channel_is_lifestyle', ' data_channel_is_entertainment', ' data_channel_is_bus', ' data_channel_is_socmed', ' data_channel_is_tech', ' data_channel_is_world', ' weekday_is_monday', ' weekday_is_tuesday', ' weekday_is_wednesday', ' weekday_is_thursday', ' weekday_is_friday', ' weekday_is_saturday', ' weekday_is_sunday']


In [4]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 特征和目标变量
X = df.drop([' shares'], axis=1) 
y = df[' shares']

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# XGBoost回归模型
xgb_reg = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

# 参数网格
param_grid = {
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'n_estimators': [100, 200],
    'gamma': [0, 0.1]
}

# 网格搜索
grid_search = GridSearchCV(
    estimator=xgb_reg,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

# 最佳参数
best_params = grid_search.best_params_
print("\n最佳参数:")
print(best_params)

# 使用最佳参数构建最终模型
final_reg = xgb.XGBRegressor(
    **best_params,
    objective='reg:squarederror',
    random_state=42
)
final_reg.fit(X_train, y_train)

# 预测与评估
y_pred = final_reg.predict(X_test)
print(f"均方误差: {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²: {r2_score(y_test, y_pred):.4f}")

Fitting 5 folds for each of 288 candidates, totalling 1440 fits

最佳参数:
{'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 100, 'subsample': 0.6}
均方误差: 169095930.56
R²: 0.0184
